# Notebook 02 — Preprocessing Data Time Series
## Early Warning System Krisis Pariwisata Bali

Notebook ini memproses:
1. Wisman Gabungan (sudah bersih)
2. Wisatawan Domestik Bulanan (format wide BPS)
3. Kurs USD/IDR
4. Daily Forex Rates (data kurs harian)
5. TPK Hotel Bintang (format wide BPS)
6. Lama Menginap Hotel Bintang (format wide BPS)
7. Inflasi Bulanan Bali

**Output:** File CSV bersih di folder `data/processed/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('✓ Library siap')

---
## 1. Wisman Gabungan — Gab_Data_Wisman_Bali.xlsx
Dataset ini sudah dalam format bersih (Tanggal, Banyak), hanya perlu rename dan convert.

In [ ]:
# Load
wisman = pd.read_excel('../data/raw/Gab_Data_Wisman_Bali.xlsx')

# Rename kolom
wisman.columns = ['date', 'wisman']

# Convert datetime
wisman['date'] = pd.to_datetime(wisman['date'])

# Sort ascending
wisman = wisman.sort_values('date').reset_index(drop=True)

# Buat kolom bulan (periode)
wisman['month'] = wisman['date'].dt.to_period('M')

# Cek missing value
print('Missing values:', wisman.isnull().sum().to_dict())
print('Rentang data:', wisman['date'].min(), 'sampai', wisman['date'].max())
print('Total baris:', len(wisman))
print()
wisman.head(5)

In [ ]:
# Simpan
wisman.to_csv('../data/processed/wisman_clean.csv', index=False)
print('✓ wisman_clean.csv disimpan')

In [ ]:
# ── Validation: Wisman Gabungan ─────────────────────────────────
print('=== VALIDATION: Wisman Gabungan ===')
print(f'Shape        : {{wisman.shape}}')
print(f'Rentang      : {{wisman["month"].min()}} -> {{wisman["month"].max()}}')
print(f'Missing      :\n{{wisman.isnull().sum().to_string()}}')
print(f'Duplikat month: {{wisman["month"].duplicated().sum()}}')
print()
print('Preview:')
print(wisman.head(3).to_string())
print()

---
## 2. Wisatawan Domestik Bulanan 2004-2025
Format BPS: wide table, baris=bulan, kolom=tahun. Perlu di-melt menjadi format panjang.

In [ ]:
path = '../data/raw/wisnus_bali_2004_2025.xlsx'
sheet_names = ['2004-2012', '2013-2021', '2022-2030']

for sheet in sheet_names:
    df = pd.read_excel(path, sheet_name=sheet, skiprows=3, header=0)
    bulan_col = df.columns[0]
    bulan_unik = df[bulan_col].dropna().unique().tolist()
    print(f"Sheet '{sheet}':")
    for b in bulan_unik:
        print(f"  '{b}'")
    print()

In [ ]:
bulan_map = {
    'Januari': '01',
    'Pebruari': '02',
    'Maret': '03',
    'April': '04',
    'Mei': '05',
    'Juni': '06',
    'Juli': '07',
    'Agustus': '08',
    'September': '09',
    'Oktober': '10',
    'Nopember': '11',
    'Desember': '12',
}

frames = []
for sheet in sheet_names:
    df = pd.read_excel(path, sheet_name=sheet, skiprows=3, header=0)
    bulan_col = df.columns[0]
    tahun_cols = [c for c in df.columns[1:] if str(c) not in ('nan', 'None', 'TOTAL')]

    df[bulan_col] = df[bulan_col].astype(str).str.strip()
    df = df[df[bulan_col].isin(bulan_map.keys())].copy()

    melted = df.melt(
        id_vars=[bulan_col],
        value_vars=tahun_cols,
        var_name='tahun',
        value_name='wisnus'
    ).rename(columns={bulan_col: 'bulan_nama'})

    frames.append(melted)
    print(f"Sheet '{sheet}': {len(melted)} rows")

df_all = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(df_all)} rows")

In [ ]:
df_all['bulan_num'] = df_all['bulan_nama'].map(bulan_map)
df_all['tahun']     = df_all['tahun'].astype(str).str[:4]
df_all['wisnus']    = pd.to_numeric(df_all['wisnus'], errors='coerce')
df_all['date']      = pd.to_datetime(
    df_all['tahun'] + '-' + df_all['bulan_num'] + '-01',
    errors='coerce'
)

wisnus_long = (
    df_all[['date', 'wisnus']]
    .dropna()
    .sort_values('date')
    .reset_index(drop=True)
)

wisnus_long.insert(1, 'month', wisnus_long['date'].dt.to_period('M'))

print(f"Shape  : {wisnus_long.shape}")
print(f"Rentang: {wisnus_long['date'].min().date()} -> {wisnus_long['date'].max().date()}")
print(f"Kolom  : {wisnus_long.columns.tolist()}")
wisnus_long.head(12)

In [ ]:
wisnus_long.to_csv('../data/processed/wisnus_clean.csv', index=False)
print('✓ wisnus_clean.csv disimpan')

In [ ]:
# ── Validation: Wisnus Domestik ─────────────────────────────────
print('=== VALIDATION: Wisnus Domestik ===')
print(f'Shape        : {{wisnus_long.shape}}')
print(f'Rentang      : {{wisnus_long["month"].min()}} -> {{wisnus_long["month"].max()}}')
print(f'Missing      :\n{{wisnus_long.isnull().sum().to_string()}}')
print(f'Duplikat month: {{wisnus_long["month"].duplicated().sum()}}')
print()
print('Preview:')
print(wisnus_long.head(3).to_string())
print()


---
## 3. Kurs USD/IDR Historical
Format: Date | Price | Open | High | Low | Change %

In [ ]:
# Load
usd = pd.read_csv('../data/raw/USD_IDR Historical Data.csv')

# Ambil kolom yang dibutuhkan
usd = usd[['Date', 'Price']].copy()
usd.columns = ['date', 'usd_idr']

# Convert date
usd['date'] = pd.to_datetime(usd['date'], format='%m/%d/%Y')

# Hapus koma dari harga 
usd['usd_idr'] = usd['usd_idr'].astype(str).str.replace(',', '')
usd['usd_idr'] = pd.to_numeric(usd['usd_idr'], errors='coerce')

# Sort ascending
usd = usd.sort_values('date').reset_index(drop=True)

# Agregasi bulanan (rata-rata kurs per bulan)
usd['month'] = usd['date'].dt.to_period('M')
monthly_usd = usd.groupby('month')['usd_idr'].mean().reset_index()
monthly_usd.columns = ['month', 'usd_idr_avg']

print('Shape harian:', usd.shape)
print('Shape bulanan:', monthly_usd.shape)
print('Rentang:', usd['date'].min(), '->', usd['date'].max())
print()
monthly_usd.tail(5)

In [ ]:
monthly_usd.to_csv('../data/processed/monthly_usd.csv', index=False)
print('✓ monthly_usd.csv disimpan')

In [ ]:
# ── Validation: Kurs USD/IDR Bulanan ─────────────────────────────────
print('=== VALIDATION: Kurs USD/IDR Bulanan ===')
print(f'Shape        : {{monthly_usd.shape}}')
print(f'Rentang      : {{monthly_usd["month"].min()}} -> {{monthly_usd["month"].max()}}')
print(f'Missing      :\n{{monthly_usd.isnull().sum().to_string()}}')
print(f'Duplikat month: {{monthly_usd["month"].duplicated().sum()}}')
print()
print('Preview:')
print(monthly_usd.head(3).to_string())
print()


---
## 4. Daily Forex Rates (Filter IDR)

In [ ]:
# Load
forex = pd.read_csv('../data/raw/daily_forex_rates.csv')

# Filter hanya IDR
forex_idr = forex[forex['currency'] == 'IDR'].copy()
forex_idr = forex_idr[['date', 'exchange_rate']].copy()

# Convert date
forex_idr['date'] = pd.to_datetime(forex_idr['date'])
forex_idr = forex_idr.sort_values('date').reset_index(drop=True)

# Agregasi bulanan
forex_idr['month'] = forex_idr['date'].dt.to_period('M')
monthly_forex = forex_idr.groupby('month')['exchange_rate'].mean().reset_index()
monthly_forex.columns = ['month', 'idr_eur_rate']

print('Shape harian:', forex_idr.shape)
print('Shape bulanan:', monthly_forex.shape)
print()
monthly_forex.tail(5)

In [ ]:
monthly_forex.to_csv('../data/processed/monthly_forex.csv', index=False)
print('✓ monthly_forex.csv disimpan')

In [ ]:
# ── Validation: Forex Daily ke Bulanan ─────────────────────────────────
print('=== VALIDATION: Forex Daily -> Bulanan ===')
print(f'Shape        : {{monthly_forex.shape}}')
print(f'Rentang      : {{monthly_forex["month"].min()}} -> {{monthly_forex["month"].max()}}')
print(f'Missing      :\n{{monthly_forex.isnull().sum().to_string()}}')
print(f'Duplikat month: {{monthly_forex["month"].duplicated().sum()}}')
print()
print('Preview:')
print(monthly_forex.head(3).to_string())
print()


---
## 5. TPK Hotel Bintang
Format BPS: baris=kelas bintang, kolom=bulan, setiap sheet=satu tahun.
Kita ambil rata-rata semua kelas bintang per bulan.

In [ ]:
excel_bintang = '../data/raw/Tingkat Penghunian Kamar (TPK) Hotel Bintang.xlsx'
excel_non_bintang = '../data/raw/Tingkat Penghunian Kamar (TPK).xlsx'

xl_b = pd.ExcelFile(excel_bintang)
xl_nb = pd.ExcelFile(excel_non_bintang)

print(f'Sheet Hotel Bintang: {xl_b.sheet_names}')
print(f'Sheet Non Bintang  : {xl_nb.sheet_names}')

kelas_valid = ['Bintang 5', 'Bintang 4', 'Bintang 3', 'Bintang 2', 'Bintang 1', 'Jumlah / Total']
kabupaten_valid = [
    'Kab. Jembrana', 'Kab. Tabanan', 'Kab. Badung', 'Kab. Gianyar',
    'Kab. Klungkung', 'Kab. Bangli', 'Kab. Karangasem', 'Kab. Buleleng',
    'Kota Denpasar', 'Provinsi Bali'
]

for label, xl in [('Hotel Bintang', xl_b), ('Non Bintang', xl_nb)]:
    raw = pd.read_excel(xl, sheet_name=xl.sheet_names[0], header=None)
    tahun = str(raw.iloc[2, 1]).strip()
    bulan = raw.iloc[3, 1:13].tolist()
    print(f'\n[{label}] Tahun: {tahun} | Bulan: {bulan}')

In [ ]:
bulan_map = {
    'Januari': '01', 'Pebruari': '02', 'Februari': '02', 'Maret': '03',
    'April': '04', 'Mei': '05', 'Juni': '06', 'Juli': '07',
    'Agustus': '08', 'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
}

# --- Proses Hotel Bintang ---
records_bintang = []
for sheet in xl_b.sheet_names:
    raw = pd.read_excel(excel_bintang, sheet_name=sheet, header=None)
    tahun_cell = str(raw.iloc[2, 1]).strip()
    bulan_headers = raw.iloc[3, 1:13].tolist()

    data = raw.iloc[4:].copy()
    data.columns = ['kelas'] + bulan_headers + ['tahunan'] + list(range(len(data.columns)-14))
    data = data[data['kelas'].astype(str).str.strip().isin(kelas_valid)].copy()

    for bulan in bulan_headers:
        bulan_str = str(bulan).strip()
        if bulan_str in bulan_map:
            values = pd.to_numeric(data[bulan], errors='coerce').dropna()
            if len(values) > 0:
                records_bintang.append({
                    'month_str': f"{tahun_cell}-{bulan_map[bulan_str]}-01",
                    'tpk_bintang': values.mean()
                })

    print(f'✓ [Bintang] Sheet {sheet} diproses')

# --- Proses Non Bintang ---
records_non_bintang = []
for sheet in xl_nb.sheet_names:
    raw = pd.read_excel(excel_non_bintang, sheet_name=sheet, header=None)
    tahun_cell = str(raw.iloc[2, 1]).strip()
    bulan_headers = raw.iloc[3, 1:13].tolist()

    data = raw.iloc[4:].copy()
    data.columns = ['kabupaten'] + bulan_headers + ['tahunan'] + list(range(len(data.columns)-14))
    data = data[data['kabupaten'].astype(str).str.strip().isin(kabupaten_valid)].copy()

    for bulan in bulan_headers:
        bulan_str = str(bulan).strip()
        if bulan_str in bulan_map:
            row_provinsi = data[data['kabupaten'].astype(str).str.strip() == 'Provinsi Bali']
            value = pd.to_numeric(row_provinsi[bulan].values[0], errors='coerce') if len(row_provinsi) > 0 else np.nan
            records_non_bintang.append({
                'month_str': f"{tahun_cell}-{bulan_map[bulan_str]}-01",
                'tpk_non_bintang': value
            })

    print(f'✓ [Non Bintang] Sheet {sheet} diproses')

# --- Merge ---
df_bintang = pd.DataFrame(records_bintang)
df_non_bintang = pd.DataFrame(records_non_bintang)

tpk_df = pd.merge(df_bintang, df_non_bintang, on='month_str', how='outer')
tpk_df['date'] = pd.to_datetime(tpk_df['month_str'])
tpk_df = tpk_df.sort_values('date').reset_index(drop=True)
tpk_df['month'] = tpk_df['date'].dt.to_period('M')
tpk_df = tpk_df[['date', 'month', 'tpk_bintang', 'tpk_non_bintang']]

print(f'\nRentang data    : {tpk_df["date"].min()} sampai {tpk_df["date"].max()}')
print(f'Total baris     : {len(tpk_df)}')
print(f'Missing bintang : {tpk_df["tpk_bintang"].isna().sum()}')
print(f'Missing non-bintang: {tpk_df["tpk_non_bintang"].isna().sum()}')

In [ ]:
tpk_df.to_csv('../data/processed/tpk_clean.csv', index=False)
print('✓ tpk_clean.csv disimpan')

In [ ]:
# ── Validation: TPK Hotel ─────────────────────────────────
print('=== VALIDATION: TPK Hotel ===')
print(f'Shape        : {{tpk_df.shape}}')
print(f'Rentang      : {{tpk_df["month"].min()}} -> {{tpk_df["month"].max()}}')
print(f'Missing      :\n{{tpk_df.isnull().sum().to_string()}}')
print(f'Duplikat month: {{tpk_df["month"].duplicated().sum()}}')
print()
print('Preview:')
print(tpk_df.head(3).to_string())
print()

---
## 6. Inflasi Bulanan Bali 2024
Format: baris=kota, kolom=bulan. Ambil baris Provinsi Bali atau rata-rata.

In [ ]:
# Load data
inflasi_raw = pd.read_excel('../data/raw/Data Inflasi.xlsx', skiprows=4)
inflasi_raw.columns = ['no', 'periode', 'inflasi', 'extra']
inflasi_raw = inflasi_raw[['no', 'periode', 'inflasi']].copy()

print('Shape raw:', inflasi_raw.shape)
print(inflasi_raw.head(5))
print(inflasi_raw.tail(5))

In [ ]:
# Mapping nama bulan Indonesia menjadi angka
bulan_map = {
    'Januari': '01', 'Februari': '02', 'Maret': '03',
    'April': '04', 'Mei': '05', 'Juni': '06', 'Juli': '07',
    'Agustus': '08', 'September': '09', 'Oktober': '10',
    'November': '11', 'Desember': '12'
}

def parse_periode(periode_str):
    """Ubah 'Desember 2025' -> '2025-12-01'"""
    parts = str(periode_str).strip().split()
    if len(parts) == 2:
        bulan_str, tahun_str = parts[0], parts[1]
        bulan_num = bulan_map.get(bulan_str)
        if bulan_num and tahun_str.isdigit():
            return f"{tahun_str}-{bulan_num}-01"
    return None

inflasi_df = inflasi_raw[inflasi_raw['periode'].notna()].copy()
inflasi_df = inflasi_df[inflasi_df['no'].apply(lambda x: str(x).isdigit() or isinstance(x, (int, float)))].copy()

inflasi_df['inflasi_processed'] = (
    inflasi_df['inflasi']
    .astype(str)
    .str.replace('%', '', regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors='coerce')
)

inflasi_df['date'] = inflasi_df['periode'].apply(parse_periode)
inflasi_df = inflasi_df[inflasi_df['date'].notna()].copy()
inflasi_df['date'] = pd.to_datetime(inflasi_df['date'])
inflasi_df['month'] = inflasi_df['date'].dt.to_period('M')

inflasi_df = inflasi_df[['date', 'month', 'inflasi_processed']].copy()

print('Shape setelah preprocessing:', inflasi_df.shape)
print('Missing values:', inflasi_df.isnull().sum().to_dict())
inflasi_df.head()

In [ ]:
# Sort ascending: Januari 2009-Desember 2025
inflasi_df = inflasi_df.sort_values('date').reset_index(drop=True)

inflasi_df = inflasi_df[
    (inflasi_df['date'].dt.year >= 2009) &
    (inflasi_df['date'].dt.year <= 2025)
].reset_index(drop=True)

print('Shape final:', inflasi_df.shape)
print('Rentang data:', inflasi_df['date'].min(), '->', inflasi_df['date'].max())
print(f'Jumlah bulan: {inflasi_df.shape[0]} (expected: 204)')
inflasi_df.to_csv('../data/processed/inflasi_clean.csv', index=False)
print('✓ inflasi_clean.csv disimpan')

---
## 7. Wisman Bali vs Indonesia Tahunan (1969–2025)
Format: dua tabel side-by-side dalam satu sheet. Baris header di row ke-4/5, data mulai row ke-6.
Berguna untuk menghitung **bali_share_pct** (porsi wisman Bali dari total wisman Indonesia).


In [ ]:
# Load raw
wisman_indo_raw = pd.read_excel(
    '../data/raw/banyaknya-wisatawan-mancanegara-ke-bali-dan-indonesia.xls',
    engine='xlrd',
    header=None
)

def parse_half_table(df, col_start, col_end):
    """Parse satu sisi tabel wisman Bali vs Indonesia."""
    sub = df.iloc[5:, col_start:col_end+1].copy()
    sub.columns = ['tahun', 'indonesia_total', 'indonesia_growth', 'bali_total', 'bali_growth']
    sub = sub.dropna(subset=['tahun'])
    sub['tahun'] = pd.to_numeric(sub['tahun'], errors='coerce')
    sub = sub.dropna(subset=['tahun'])
    sub['tahun'] = sub['tahun'].astype(int)
    sub['indonesia_total'] = pd.to_numeric(sub['indonesia_total'], errors='coerce')
    sub['bali_total'] = pd.to_numeric(sub['bali_total'], errors='coerce')
    return sub[['tahun', 'indonesia_total', 'bali_total']]

left  = parse_half_table(wisman_indo_raw, 0, 4)
right = parse_half_table(wisman_indo_raw, 6, 10)
wisman_indo_annual = pd.concat([left, right], ignore_index=True)
wisman_indo_annual = wisman_indo_annual.sort_values('tahun').reset_index(drop=True)

wisman_indo_annual['bali_share_pct'] = (
    wisman_indo_annual['bali_total'] / wisman_indo_annual['indonesia_total'] * 100
).round(2)

print('Shape:', wisman_indo_annual.shape)
print('Rentang tahun:', wisman_indo_annual['tahun'].min(), '-', wisman_indo_annual['tahun'].max())
print()
wisman_indo_annual.tail(10)


In [ ]:
# Simpan inflasi
inflasi_df.to_csv('../data/processed/inflasi_clean.csv', index=False)
print('✓ inflasi_clean.csv tersimpan')
print(f'   Shape  : {inflasi_df.shape}')
print(f'   Rentang: {inflasi_df["month"].min()} -> {inflasi_df["month"].max()}')
print(f'   Missing: {inflasi_df.isnull().sum().sum()}')

In [ ]:
wisman_indo_annual.to_csv('../data/processed/wisman_vs_indonesia_clean.csv', index=False)
print('✓ wisman_vs_indonesia_clean.csv disimpan')
print('Kolom:', wisman_indo_annual.columns.tolist())
print('Catatan: Data tahunan — akan di-join ke timeline bulanan berdasarkan tahun di NB04')

In [ ]:
# ── Validation: Wisman Bali vs Indonesia ─────────────────────────────────
print('=== VALIDATION: Wisman Bali vs Indonesia ===')
print(f'Shape        : {{wisman_indo_annual.shape}}')
print(f'Rentang      : {{wisman_indo_annual["month"].min()}} -> {{wisman_indo_annual["month"].max()}}')
print(f'Missing      :\n{{wisman_indo_annual.isnull().sum().to_string()}}')
print(f'Duplikat month: {{wisman_indo_annual["month"].duplicated().sum()}}')
print()
print('Preview:')
print(wisman_indo_annual.head(3).to_string())
print()

---
## 8. Cek Semua Output

In [ ]:
import os

processed_files = os.listdir('../data/processed/')
print('File di ../data/processed/:')
for f in sorted(processed_files):
    if f.endswith('.csv'):
        df = pd.read_csv(f'../data/processed/{f}')
        print(f'  ✓ {f} — {df.shape[0]} baris, {df.shape[1]} kolom')

## Coverage Validation — Semua Output Processed

In [ ]:
# ── Coverage Validation — Semua File Processed ─────────────
import os, pandas as pd

print('=== COVERAGE VALIDATION SEMUA FILE PROCESSED ===')
print()

processed = {
    'wisman_clean.csv'          : 'month',
    'wisnus_clean.csv'          : 'month',
    'monthly_usd.csv'           : 'month',
    'tpk_clean.csv'             : 'month',
    'inflasi_clean.csv'         : 'month',
    'wisman_vs_indonesia_clean.csv': 'year',
}

for fname, date_col in processed.items():
    path = f'../data/processed/{fname}'
    if os.path.exists(path):
        df  = pd.read_csv(path)
        mn  = df[date_col].min() if date_col in df.columns else 'N/A'
        mx  = df[date_col].max() if date_col in df.columns else 'N/A'
        mis = df.isnull().sum().sum()
        print(f'  {fname:40s} | {mn} -> {mx} | missing={mis}')
    else:
        print(f'  {fname:40s} | FILE TIDAK ADA')

print()
print('✓ Coverage check selesai.')
